In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [4]:
saxpy=r"""

#include <cstdio>
#include <cuda_runtime.h>

__global__ void saxpy(int n, float a, const float* x, float* y){
    int i=blockIdx.x*blockDim.x+threadIdx.x;
    if (i<n){
        y[i]=a*x[i]+y[i];
    }
}

int main(){
    const int n=1<<20;
    size_t bytes=n*sizeof(float);

    float *x, *y;
    cudaMallocManaged(&x, bytes);
    cudaMallocManaged(&y, bytes);

    for(int i=0; i<n; i++){
    x[i]=1.0f;
    y[i]=2.0f;
    }

    int threads=256;
    int blocks=(n+threads-1)/threads;
    saxpy<<<blocks, threads>>>(n, 3.0f, x, y);
    cudaDeviceSynchronize();

    for(int i=0; i<5; i++){
    printf("y[%d]=%f\n", i, y[i]);
}

    cudaFree(x);
    cudaFree(y);
    return 0;
}

"""

with open('saxpy.cu', 'w') as f:
    f.write(saxpy)

In [5]:
!nvcc -arch=sm_75 saxpy.cu -o saxpy

In [6]:
!./saxpy

y[0]=5.000000
y[1]=5.000000
y[2]=5.000000
y[3]=5.000000
y[4]=5.000000
